<div style="background:#1E1E1E;border:1px solid #333333;color:#D4D4D4;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif;">
  <div style="font-size:13px;letter-spacing:2px;color:#4DD0E1;font-weight:700;">UNIVERSIDAD AUT&Oacute;NOMA DE GUADALAJARA</div>
  <div style="font-size:26px;font-weight:700;margin-top:6px;color:#FFFFFF;">Examen — Primer Parcial · SOLUCIÓN</div>
  <div style="font-size:16px;margin-top:4px;color:#A0A0A0;">Unidad 3 (Tuberías de aprendizaje automático) y Unidad 4 (Ingeniería de funciones)</div>
  <div style="margin-top:14px;font-size:14px;">
    <span style="background:rgba(232, 145, 43, 0.2);color:#F5B041;border:1px solid #E8912B;padding:4px 12px;border-radius:14px;font-weight:700;">FP13 &middot; Inteligencia Artificial</span>
    <span style="background:rgba(15, 163, 177, 0.2);color:#4DD0E1;border:1px solid #0FA3B1;padding:4px 12px;border-radius:14px;font-weight:700;margin-left:8px;">90 minutos</span>
    <span style="background:#2D2D2D;color:#E0E0E0;border:1px solid #444444;padding:4px 12px;border-radius:14px;font-weight:700;margin-left:8px;">100 puntos</span>
  </div>
</div>

**Alumno:** _______________________________  **Matrícula:** ____________  **Fecha:** ____________

<br>

---

### Instrucciones

1. El examen es **enteramente práctico**. Se resuelve dentro de este notebook.
2. Las celdas marcadas **`CELDA DADA`** ya están resueltas: **ejecútalas sin modificarlas**. Contienen el diagnóstico que necesitas para decidir.
3. Las celdas marcadas **`TU TURNO`** son tuyas. Escribe el código de la decisión.
4. Después de cada punto hay una celda de **justificación obligatoria**. Una decisión correcta sin evidencia que la respalde vale **la mitad**.
5. Ejecuta el notebook **de principio a fin, en orden**. Si una celda falla, las siguientes fallarán.

<div style="border-left:5px solid #0FA3B1;background:rgba(15, 163, 177, 0.08);color:#d4d4d4;padding:12px 16px;border-radius:5px;font-family:Calibri,Arial,sans-serif;">
<b style="color:#4DD0E1;">Cómo se califica cada punto</b><br><b>Ejecución (50 %)</b>: el código corre y hace lo que dice.<br><b>Decisión (30 %)</b>: la elección es la que el dato exige.<br><b>Justificación (20 %)</b>: citas la evidencia numérica concreta.<br>
</div>

---

### Contexto clínico

Trabajas con el registro de una **cohorte de síndrome coronario agudo (SICA)** de un hospital de tercer nivel.
Cada fila es un **ingreso hospitalario**. El objetivo es predecir `reingreso_30d`: si el paciente reingresa
dentro de los 30 días posteriores al alta.

### Diccionario de datos

| Variable | Tipo | Descripción | Momento del registro |
|---|---|---|---|
| `id_paciente` | Texto | Identificador del paciente | — |
| `edad` | Numérica | Años cumplidos | Ingreso |
| `sexo` | Categórica nominal | M / F | Ingreso |
| `clase_nyha` | Categórica **ordinal** | Clase funcional NYHA: I < II < III < IV | Ingreso |
| `estadio_erc` | Categórica **ordinal** | Estadio de enfermedad renal crónica (KDIGO): 1 < 2 < 3 < 4 < 5 | Ingreso |
| `servicio_ingreso` | Categórica nominal | Unidad médica de admisión (código interno) | Ingreso |
| `presion_sistolica` | Numérica | Presión arterial sistólica (mmHg) | Ingreso |
| `fevi_ingreso` | Numérica | Fracción de eyección del ventrículo izquierdo (%) | Ingreso |
| `creatinina` | Numérica | Creatinina sérica (mg/dL) | Ingreso |
| `dimero_d` | Numérica | Dímero D (ng/mL) | Ingreso |
| `delta_fevi` | Numérica | Cambio en la fracción de eyección respecto al estudio previo (**puntos porcentuales, puede ser negativo**) | Ingreso |
| `pro_bnp` | Numérica | NT-proBNP (pg/mL). **Se solicita a criterio médico, no de forma sistemática** | Ingreso |
| `disnea_ingreso` | Binaria | Disnea presente al ingreso (0/1) | Ingreso |
| `dias_estancia_uci` | Numérica | Días de estancia en la Unidad de Cuidados Intensivos. **Se captura al cierre del expediente, en el momento del alta** | **Alta** |
| `reingreso_30d` | Binaria | **Variable objetivo**: reingreso a 30 días (0/1) | Seguimiento |


In [ ]:
# ============================================================
#  CELDA DADA — ejecútala sin modificarla
# ============================================================
import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, recall_score)

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.figsize": (11, 3.2), "axes.grid": True,
                     "grid.alpha": .3, "font.size": 9})

SEMILLA = 42


def lambda_ic(x, alpha=0.05):
    """Lambda de Yeo-Johnson por maxima verosimilitud + IC 95 % (verosimilitud perfilada).

    Devuelve (lambda, limite_inferior, limite_superior).
    Si el IC contiene 1.0, la transformacion NO es estadisticamente necesaria.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    lam = stats.yeojohnson_normmax(x)
    corte = stats.yeojohnson_llf(lam, x) - 0.5 * stats.chi2.ppf(1 - alpha, 1)
    malla = np.linspace(lam - 6, lam + 6, 2001)
    llf = np.array([stats.yeojohnson_llf(l, x) for l in malla])
    dentro = malla[llf >= corte]
    return lam, dentro.min(), dentro.max()


def perfil(df, columnas):
    """Tabla diagnostica: sesgo, minimo, negativos, ceros, lambda e IC 95 %."""
    filas = []
    for c in columnas:
        s = df[c].dropna().astype(float)
        lam, lo, hi = lambda_ic(s)
        filas.append({
            "variable": c,
            "sesgo": round(float(stats.skew(s)), 2),
            "min": round(float(s.min()), 2),
            "negativos": int((s < 0).sum()),
            "ceros": int((s == 0).sum()),
            "lambda": round(lam, 3),
            "IC_inf": round(lo, 2),
            "IC_sup": round(hi, 2),
            "IC_contiene_1": bool(lo <= 1 <= hi),
        })
    return pd.DataFrame(filas)


def cargar(nombre):
    """Busca el CSV junto al notebook o en ./datos/."""
    for r in (nombre, os.path.join("datos", nombre),
              os.path.join("..", "datos", nombre)):
        if os.path.exists(r):
            return pd.read_csv(r)
    raise FileNotFoundError(
        f"No encuentro '{nombre}'. Colocalo en la misma carpeta que este notebook.")


print("Entorno listo.")

In [ ]:
df = cargar("cohorte_sica_examen.csv")
print("Dimensiones:", df.shape)
df.head()

---
## Punto 1 — Integridad de los registros y segregación de datos &nbsp;&nbsp;`12 pts`
*Subtemas 3.2 y 3.4*

In [ ]:
# ============================ CELDA DADA ============================
print("Filas (ingresos):     ", len(df))
print("Pacientes únicos:     ", df["id_paciente"].nunique())
print("Diferencia:           ", len(df) - df["id_paciente"].nunique())
print()
print("Pacientes con más de un ingreso (primeros 5):")
print(df["id_paciente"].value_counts().head(5).to_string())
print()
pac = df.groupby("id_paciente", as_index=False)["reingreso_30d"].max()
print("Resumen a nivel PACIENTE -> filas:", len(pac),
      "| prevalencia: %.4f" % pac["reingreso_30d"].mean())

**Pregunta.** Construye el conjunto de entrenamiento (`df_tr`) y el de prueba (`df_te`) con
`test_size=0.25`, `random_state=SEMILLA` y estratificando por el *`reingreso_30d`*.

Antes de escribir, mira el diagnóstico de arriba y decide **sobre qué unidad debe hacerse la partición**.
La tabla `pac` ya está construida por si te resulta útil.

In [ ]:
# ============================= TU TURNO =============================
# Objetivo: df_tr y df_te
# ---- ESCRIBE TU CÓDIGO AQUÍ ----

df_tr = None
df_te = None

In [ ]:
# ============================ CELDA DADA ============================
# Verificación de tu partición
solapamiento = set(df_tr["id_paciente"]) & set(df_te["id_paciente"])
print("Filas train / test:        ", len(df_tr), "/", len(df_te))
print("Pacientes compartidos:     ", len(solapamiento), "  <-- debe ser 0")
print("Prevalencia train:          %.4f" % df_tr["reingreso_30d"].mean())
print("Prevalencia test:           %.4f" % df_te["reingreso_30d"].mean())


> **Tu justificación (obligatoria):**
>
> _Escribe aquí. Cita la evidencia numérica concreta que sustenta tu decisión._


---
## Punto 2 — Selección de funciones: variables admisibles &nbsp;&nbsp;`8 pts`
*Subtemas 3.1 y 4.1*

In [ ]:
# ============================ CELDA DADA ============================
numericas = df_tr.select_dtypes(include=np.number).columns.drop("reingreso_30d")
corr = df_tr[numericas].corrwith(df_tr["reingreso_30d"]).sort_values(key=abs, ascending=False)
print("Correlación de cada variable numérica con el desenlace:")
print(corr.round(3).to_string())

**Pregunta.** Una de estas variables **no puede formar parte del modelo**. Identifícala,
elimínala de `df_tr` y de `df_te`, y explica por qué.

> Pista: vuelve a leer la columna *"Momento del registro"* del diccionario de datos.

In [ ]:
# ============================= TU TURNO =============================
# ---- ESCRIBE TU CÓDIGO AQUÍ ----



> **Tu justificación (obligatoria):**
>
> _Escribe aquí. Cita la evidencia numérica concreta que sustenta tu decisión._


---
## Punto 3 — Valores faltantes &nbsp;&nbsp;`12 pts`
*Subtema 3.3*

In [ ]:
# ============================ CELDA DADA ============================
print("Porcentaje de faltantes por variable (entrenamiento):")
print((df_tr.isna().mean() * 100).round(1).loc[lambda s: s > 0].to_string())
print()
print("Faltantes de pro_bnp según disnea al ingreso:")
tabla = df_tr.groupby("disnea_ingreso")["pro_bnp"].agg(
    n="size", faltantes=lambda s: s.isna().sum(), pct_faltante=lambda s: round(s.isna().mean() * 100, 1))
print(tabla.to_string())

**Pregunta.** Trata los faltantes de `pro_bnp`. Antes de imputar, observa la tabla de arriba:
el patrón de ausencia **no es aleatorio**.

Tu solución debe (a) preservar la información contenida en el hecho mismo de que el estudio no se haya
solicitado, y (b) rellenar el valor numérico **sin usar información del conjunto de prueba**.

In [ ]:
# ============================= TU TURNO =============================
# ---- ESCRIBE TU CÓDIGO AQUÍ ----



> **Tu justificación (obligatoria):**
>
> _Escribe aquí. Cita la evidencia numérica concreta que sustenta tu decisión._


---
## Punto 4 — Gestión de valores atípicos &nbsp;&nbsp;`14 pts`
*Subtema 4.3*

In [ ]:
# ============================ CELDA DADA ============================
def limites_iqr(s):
    q1, q3 = s.quantile([.25, .75]); r = q3 - q1
    return q1 - 1.5 * r, q3 + 1.5 * r

for v in ["creatinina", "presion_sistolica"]:
    lo, hi = limites_iqr(df_tr[v])
    n = int(((df_tr[v] < lo) | (df_tr[v] > hi)).sum())
    print(f"{v:20s} límites IQR = [{lo:7.2f}, {hi:7.2f}]   atípicos = {n:4d}   mínimo = {df_tr[v].min():.2f}")

print("\ncreatinina por estadio de enfermedad renal crónica (KDIGO):")
print(df_tr.groupby("estadio_erc")["creatinina"].agg(["size", "median", "max"]).round(2).to_string())

print("\npresion_sistolica -> registros en cero:", int((df_tr["presion_sistolica"] == 0).sum()))
fig, ax = plt.subplots(1, 2, figsize=(11, 2.8))
ax[0].boxplot(df_tr["creatinina"], vert=False); ax[0].set_title("creatinina (mg/dL)")
ax[1].hist(df_tr["presion_sistolica"], bins=40, color="#1E2664"); ax[1].set_title("presion_sistolica (mmHg)")
plt.tight_layout(); plt.show()

**Pregunta.** El criterio IQR marca valores atípicos en **ambas** variables.
**Solo una de las dos debe corregirse.** Decide cuál, aplica el tratamiento, y explica por qué la otra
se deja intacta.

In [ ]:
# ============================= TU TURNO =============================
# ---- ESCRIBE TU CÓDIGO AQUÍ ----



> **Tu justificación (obligatoria):**
>
> _Escribe aquí. Cita la evidencia numérica concreta que sustenta tu decisión._


---
## Punto 5 — Codificación de variables categóricas &nbsp;&nbsp;`14 pts`
*Subtemas 4.4 y 4.7*

In [ ]:
# ============================ CELDA DADA ============================
for v in ["sexo", "clase_nyha", "estadio_erc", "servicio_ingreso"]:
    print(f"{v:20s} niveles = {df_tr[v].nunique():3d}")
print()
print("clase_nyha:"); print(df_tr["clase_nyha"].value_counts().to_string())
print()
print("servicio_ingreso -> 8 niveles más frecuentes de", df_tr["servicio_ingreso"].nunique(), ":")
print(df_tr["servicio_ingreso"].value_counts().head(8).to_string())

**Pregunta.** Codifica las cuatro variables categóricas. **No todas admiten el mismo tratamiento.**
Revisa en el diccionario cuáles son nominales y cuáles ordinales, y observa la cardinalidad.

Crea las columnas nuevas con estos nombres para que la celda de ensamblado funcione:
`nyha_ord`, `servicio_grp` (las demás las genera `get_dummies`).

In [ ]:
# ============================= TU TURNO =============================
# ---- ESCRIBE TU CÓDIGO AQUÍ ----

# nyha_ord      -> ...
# estadio_erc   -> ...
# servicio_grp  -> ...
# sexo          -> ...

In [ ]:
# ============================ CELDA DADA ============================
# Ensamblado del one-hot para las variables NOMINALES y alineación de columnas
df_tr = pd.get_dummies(df_tr, columns=["sexo", "servicio_grp"], drop_first=True, dtype=int)
df_te = pd.get_dummies(df_te, columns=["sexo", "servicio_grp"], drop_first=True, dtype=int)
df_te = df_te.reindex(columns=df_tr.columns, fill_value=0)   # el test debe tener las mismas columnas
print("Columnas tras codificar -> train:", df_tr.shape[1], "| test:", df_te.shape[1])


> **Tu justificación (obligatoria):**
>
> _Escribe aquí. Cita la evidencia numérica concreta que sustenta tu decisión._


---
## Punto 6 — Corrección del sesgo &nbsp;&nbsp;`20 pts`
*Subtema 4.5 — el punto de mayor peso del examen*

<div style="border-left:5px solid #0FA3B1;background:rgba(15, 163, 177, 0.08);color:#d4d4d4;padding:12px 16px;border-radius:5px;font-family:Calibri,Arial,sans-serif;">
<b style="color:#4DD0E1;">Recordatorio de criterio</b><br>El logaritmo exige valores <b>estrictamente positivos</b>.<br>Yeo-Johnson admite ceros y negativos, y estima &lambda; por m&aacute;xima verosimilitud.<br>&lambda; &lt; 1 corrige sesgo <b>derecho</b>; &lambda; &gt; 1 corrige sesgo <b>izquierdo</b>.<br>Si el <b>IC 95 % de &lambda; contiene 1.0</b>, la transformaci&oacute;n no es estad&iacute;sticamente necesaria.
</div>

In [ ]:
# ============================ CELDA DADA ============================
objetivo = ["pro_bnp", "dimero_d", "delta_fevi", "fevi_ingreso", "edad"]
print(perfil(df_tr, objetivo).to_string(index=False))

fig, ax = plt.subplots(1, 5, figsize=(14, 2.6))
for a, v in zip(ax, objetivo):
    a.hist(df_tr[v], bins=35, color="#0FA3B1")
    a.set_title(f"{v}\nsesgo = {stats.skew(df_tr[v].dropna()):+.2f}", fontsize=8)
plt.tight_layout(); plt.show()

**Pregunta.** Las cinco variables presentan situaciones **distintas**. Decide qué hacer con cada una.

La primera (`pro_bnp`) **ya viene resuelta como ejemplo**: úsala como modelo del formato esperado.
Resuelve las **cuatro restantes**.

> Advertencia: `PowerTransformer` estandariza por omisión. Usa `standardize=False`, porque el escalamiento
> es una decisión aparte que tomarás en el Punto 7.

In [ ]:
# ============================ CELDA DADA — EJEMPLO RESUELTO ============================
# pro_bnp: sesgo muy positivo, mínimo > 0, sin ceros ni negativos  ->  LOGARITMO
df_tr["pro_bnp"] = np.log(df_tr["pro_bnp"])
df_te["pro_bnp"] = np.log(df_te["pro_bnp"])
print("pro_bnp -> log | sesgo resultante: %+.2f" % stats.skew(df_tr["pro_bnp"]))

In [ ]:
# ============================= TU TURNO =============================
# dimero_d      -> ?
# delta_fevi    -> ?
# fevi_ingreso  -> ?
# edad          -> ?
# ---- ESCRIBE TU CÓDIGO AQUÍ ----


> **Tu justificación (obligatoria).** Completa la tabla:
>
> | Variable | Decisión | Evidencia numérica que la sustenta |
> |---|---|---|
> | `dimero_d` | | |
> | `delta_fevi` | | |
> | `fevi_ingreso` | | |
> | `edad` | | |


---
## Punto 7 — Escalamiento &nbsp;&nbsp;`10 pts`
*Subtema 4.6*

In [ ]:
# ============================ CELDA DADA ============================
num_final = ["edad", "presion_sistolica", "fevi_ingreso", "creatinina",
             "dimero_d", "delta_fevi", "pro_bnp", "nyha_ord", "estadio_erc"]
print(df_tr[num_final].describe().T[["mean", "std", "min", "50%", "max"]].round(2).to_string())
print("\nSesgo residual de cada variable:")
print(df_tr[num_final].apply(lambda s: round(float(stats.skew(s)), 2)).to_string())

**Pregunta.** Escala las variables numéricas. **No todas deben recibir el mismo escalador.**

Recuerda la decisión que tomaste en el Punto 4: hay una variable cuyos valores extremos **conservaste
deliberadamente** porque son clínicamente válidos. Un escalador basado en media y desviación estándar
se ve arrastrado por ellos.

Deja el resultado en `df_tr[num_final]` y `df_te[num_final]`.

In [ ]:
# ============================= TU TURNO =============================
# ---- ESCRIBE TU CÓDIGO AQUÍ ----



> **Tu justificación (obligatoria):**
>
> _Escribe aquí. Cita la evidencia numérica concreta que sustenta tu decisión._


---
## Punto 8 — Entrenamiento y elección de la métrica &nbsp;&nbsp;`10 pts`
*Subtemas 3.5 y 3.1*

In [ ]:
# ============================ CELDA DADA ============================
descartar = ["id_paciente", "reingreso_30d", "clase_nyha", "servicio_ingreso"]
X_tr = df_tr.drop(columns=[c for c in descartar if c in df_tr.columns])
X_te = df_te.drop(columns=[c for c in descartar if c in df_te.columns])
y_tr, y_te = df_tr["reingreso_30d"], df_te["reingreso_30d"]

assert X_tr.select_dtypes(exclude=np.number).empty, "Quedan columnas no numéricas en X_tr"
print("Matriz final -> train:", X_tr.shape, "| test:", X_te.shape)
print("Prevalencia en prueba: %.4f" % y_te.mean())

modelo = LogisticRegression(solver="lbfgs", max_iter=3000, random_state=SEMILLA).fit(X_tr, y_tr)
pred = modelo.predict(X_te)
prob = modelo.predict_proba(X_te)[:, 1]

print("\n--- MODELO A (sin ponderar) -------------------------------------")
print("Accuracy : %.4f" % accuracy_score(y_te, pred))
print("Recall   : %.4f" % recall_score(y_te, pred, zero_division=0))
print("AUC ROC  : %.4f" % roc_auc_score(y_te, prob))
print("Matriz de confusión [fila = real, columna = predicho]:")
print(confusion_matrix(y_te, pred))

modelo_b = LogisticRegression(solver="lbfgs", max_iter=3000, class_weight="balanced",
                              random_state=SEMILLA).fit(X_tr, y_tr)
pred_b = modelo_b.predict(X_te)
prob_b = modelo_b.predict_proba(X_te)[:, 1]

print("\n--- MODELO B (class_weight='balanced') --------------------------")
print("Accuracy : %.4f" % accuracy_score(y_te, pred_b))
print("Recall   : %.4f" % recall_score(y_te, pred_b, zero_division=0))
print("AUC ROC  : %.4f" % roc_auc_score(y_te, prob_b))
print("Matriz de confusión:")
print(confusion_matrix(y_te, pred_b))

print("\n--- Referencia: clasificador trivial 'nadie reingresa' ----------")
print("Accuracy : %.4f" % accuracy_score(y_te, np.zeros_like(y_te)))

**Pregunta.** Responde las tres, en prosa, con los números de la salida anterior:

1. El Modelo A alcanza una accuracy alta. ¿Es un buen modelo? Compáralo con el clasificador trivial.
2. ¿Qué métrica reportarías al comité clínico del hospital y por qué?
3. ¿Cuál de los dos modelos entregarías? Justifica en términos del **costo clínico** de cada tipo de error.

> **Tus respuestas:**
>
> **1.** _Escribe aquí._
>
> **2.** _Escribe aquí._
>
> **3.** _Escribe aquí._


---
<div style="border-left:5px solid #E8912B;background:rgba(232, 145, 43, 0.08);color:#d4d4d4;padding:12px 16px;border-radius:5px;font-family:Calibri,Arial,sans-serif;">
<b style="color:#F5B041;">Antes de entregar</b><br>1. <b>Reinicia el kernel y ejecuta todo de nuevo</b> (Kernel &rarr; Restart &amp; Run All). El notebook debe correr de principio a fin sin errores.<br>2. Verifica que las ocho celdas de justificaci&oacute;n est&eacute;n escritas. Una decisi&oacute;n sin justificaci&oacute;n vale la mitad.<br>3. Guarda como <code>Parcial1_ApellidoNombre.ipynb</code> y s&uacute;belo a Mis Cursos.
</div>